# MedScan — Comparison Training (Kaggle)
This notebook trains and compares three different CNN backbones (ResNet18, EfficientNet-B5, DenseNet121) on the preprocessed MedScan 2D slice dataset. 
It splits the training set to create a validation set, calculates class weights, and prints the exact dataset sizes before beginning training.


In [ ]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Paths & Setup


In this section, we set up the environment paths. We use an automated path-detection mechanism to seamlessly transition between local development (Windows) and Kaggle kernel environments. This ensures our training pipeline is environment-agnostic and robust.

In [ ]:
# Automatically detect if running on Kaggle or locally
kaggle_path = "/kaggle/input/medscan-processed-slices/Processed_2D_Slices"
local_path = r"c:/Projects/MedScan-Alzheimer's/data/processed/Processed_2D_Slices"

if os.path.exists(kaggle_path):
    DATA_DIR = kaggle_path
    print("Detected Kaggle environment.")
elif os.path.exists(local_path):
    DATA_DIR = local_path
    print("Detected Local Windows environment.")
else:
    raise FileNotFoundError("Could not find Processed_2D_Slices folder. Please check your path.")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

assert os.path.exists(TRAIN_DIR), f"Train dir not found: {TRAIN_DIR}"
assert os.path.exists(TEST_DIR), f"Test dir not found: {TEST_DIR}"

BATCH_SIZE = 32
NUM_CLASSES = 3
IMG_SIZE = 224


## 1.5 Offline Data Augmentation (Class Balancing)
To explicitly increase the total number of training inputs and balance the classes before loading them into PyTorch, we generate augmented copies of images in the minority classes (`CDR_0.5` and `CDR_1.0`) and save them directly to disk. This ensures `len(train_dataset)` physically increases.

Medical datasets often suffer from severe class imbalance. To counteract this without causing data leakage, we perform **offline data augmentation**. By applying random rotations, flips, and color jitter to minority classes *before* the PyTorch DataLoader accesses them, we physically balance the dataset on disk. This prevents the model from developing a majority-class bias.

In [ ]:
from PIL import Image
import glob
import os
import shutil
from torchvision import transforms

# On Kaggle, /kaggle/input is READ-ONLY.
# We must copy the training data to the writable /kaggle/working/ directory before augmenting.
WRITABLE_TRAIN_DIR = TRAIN_DIR
if '/kaggle/input' in TRAIN_DIR:
    WRITABLE_TRAIN_DIR = '/kaggle/working/train_augmented'
    if not os.path.exists(WRITABLE_TRAIN_DIR):
        print(f"Copying dataset to writable directory: {WRITABLE_TRAIN_DIR}...")
        shutil.copytree(TRAIN_DIR, WRITABLE_TRAIN_DIR)
    TRAIN_DIR = WRITABLE_TRAIN_DIR # Update for the DataLoader!

# Only augment the training set
offline_transforms = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2)
])

# Find the maximum class size to balance against
class_counts = {}
for cls in os.listdir(TRAIN_DIR):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        class_counts[cls] = len(glob.glob(os.path.join(cls_path, '*.png')))

max_count = max(class_counts.values())
print(f"Original class counts: {class_counts}")
print(f"Target size for minority classes: {max_count}\n")

for cls, count in class_counts.items():
    if count < max_count:
        needed = max_count - count
        print(f"Generating {needed} augmented images for {cls}...")
        
        cls_path = os.path.join(TRAIN_DIR, cls)
        images = glob.glob(os.path.join(cls_path, '*.png'))
        
        for i in range(needed):
            # Randomly sample an existing image
            img_path = random.choice(images)
            img = Image.open(img_path)
            
            # Apply offline augmentation
            aug_img = offline_transforms(img)
            
            # Save as a new file in the writable directory
            new_filename = f"aug_{i}_{os.path.basename(img_path)}"
            aug_img.save(os.path.join(cls_path, new_filename))

print("\nOffline augmentation complete! The dataset is now physically balanced.")


## 2. Dataset Loading & Size Tracking


We define our PyTorch `transforms` and construct our `DataLoader` objects. Note that we apply random augmentations *only* to the training set to prevent overfitting. The validation and test sets remain unaltered to provide an unbiased evaluation metric.

In [ ]:
# Define Transforms
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor()
])

# Validation and Test do NOT get augmentations, only ToTensor
# Normalization was already applied in preprocessing!
test_transforms = transforms.Compose([
    transforms.ToTensor()
])

# Load datasets
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transforms)

class_names = full_train_dataset.classes
print(f"Detected Classes: {class_names}")

# Split full_train into train (80%) and val (20%)
val_size = int(0.2 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

# Override the transform for the validation subset so it doesn't get random augmentations
val_dataset.dataset.transform = test_transforms

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("\n" + "="*40)
print("DATASET SIZE TRACKING")
print("="*40)
print(f"Total training images    : {len(train_dataset)} (Batches: {len(train_loader)})")
print(f"Total validation images  : {len(val_dataset)} (Batches: {len(val_loader)})")
print(f"Total testing images     : {len(test_dataset)} (Batches: {len(test_loader)})")
print("="*40 + "\n")


## 3. Class Weights Calculation


Even with augmented data, minor imbalances can persist. We utilize Scikit-Learn's `compute_class_weight` to calculate balanced penalty weights for our `CrossEntropyLoss` function. This heavily penalizes the model for misclassifying minority examples, forcing it to learn features for all classes equally.

In [ ]:
# Get labels from the training subset to compute weights
# Since random_split returns a Subset, we extract the original targets using the subset indices
train_targets = [full_train_dataset.targets[i] for i in train_dataset.indices]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_targets),
    y=train_targets
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Calculated Class Weights (to handle imbalance):")
for name, weight in zip(class_names, class_weights):
    print(f"  {name}: {weight:.4f}")

# Define the Loss Function with these weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)


## 4. Model Architectures & Training Loop


Here we define our training loop and experiment with three different highly-regarded CNN architectures: **ResNet18**, **EfficientNet-B5**, and **DenseNet121**. 

We use **Early Stopping** based on the Validation Macro-F1 score to halt training if the model stops improving, preventing overfitting. The best weights are restored at the end of training.

In [ ]:
def get_model(model_name, num_classes=3):
    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        # ResNet takes 3 channels, our data is grayscale saved as PNG (RGB). It will work out of the box.
    elif model_name == "efficientnet_b5":
        model = models.efficientnet_b5(weights=models.EfficientNet_B5_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    
    return model.to(DEVICE)

def train_model(model, model_name, num_epochs=15, patience=3):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_f1 = 0.0
    epochs_no_improve = 0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print(f"\n--- Training {model_name} ---")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # TRAIN PHASE
        model.train()
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        train_loss = running_loss / len(train_dataset)
        train_acc = running_corrects.double() / len(train_dataset)
        
        # VAL PHASE
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)
                
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        val_loss = val_loss / len(val_dataset)
        val_acc = val_corrects.double() / len(val_dataset)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc.item())
        history['val_acc'].append(val_acc.item())
        
        print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
        
        # Early Stopping check
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            # Save checkpoint locally
            torch.save(model.state_dict(), f"{model_name}_best.pt")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
                
    time_elapsed = time.time() - start_time
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Val Macro F1: {best_val_f1:4f}")
    
    # Load best weights
    model.load_state_dict(best_model_wts)
    return model, history


## 5. Evaluation & Comparison


Finally, we evaluate all three models on the unseen test dataset. We generate classification reports and Confusion Matrices to visually compare their true-positive and false-positive rates across all dementia severity levels. This helps us select the ultimate backbone for deployment.

In [ ]:
def evaluate_model(model, model_name):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print(f"\n{'='*40}")
    print(f"TEST EVALUATION: {model_name}")
    print(f"{'='*40}")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# List of models you requested
models_to_test = ["resnet18", "efficientnet_b5", "densenet121"]
trained_models = {}

for m_name in models_to_test:
    model_ft = get_model(m_name, NUM_CLASSES)
    model_ft, hist = train_model(model_ft, m_name, num_epochs=15, patience=3)
    trained_models[m_name] = model_ft
    evaluate_model(model_ft, m_name)
